<a href="https://colab.research.google.com/github/massimilianogasparini-author/creative-loop-dynamics/blob/main/mad_collapse_simulator_creative_loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

# --- 1. DEFINIZIONE DELLE COSTANTI TERMODINAMICHE E INFORMAZIONALI ---
# Valori fissi come derivati dall'analisi dimensionale del sistema
K = 1.0       # Capacità di carico (saturazione) dell'informazione umana
K_s = 1.2     # Capacità di carico (overfitting) dell'informazione sintetica
gamma = 0.3   # Tasso intrinseco di crescita dell'IA (Dinamica di Gompertz)
lam = 0.1     # Fattore di soppressione umana sui dati sintetici
h_tau = 0.1   # Soglia MAD (Model Autophagy Disorder) - Orizzonte degli eventi

# --- 2. SISTEMA DI EQUAZIONI DIFFERENZIALI ORDINARIE (ODE) ---
def creative_loop_system(t, state, alpha, beta, phi):
    """
    Definisce il sistema di ODE accoppiate per h(t) e s(t).
    """
    h, s = state

    # Clamp per stabilità numerica: impedisce a h e s di scendere a zero assoluto
    # o a valori negativi, evitando divisioni per zero o logaritmi indefiniti.
    h = max(h, 1e-6)
    s = max(s, 1e-6)

    # Equazione 1: Dinamica dell'Informazione Umana
    dh_dt = alpha * h * (1.0 - h / K) - beta * h * s + phi

    # Equazione 2: Dinamica dell'Informazione Sintetica
    ds_dt = gamma * s * np.log(K_s / s) - lam * s * h

    return [dh_dt, ds_dt]

# --- 3. MOTORE DI SIMULAZIONE E RENDERING GRAFICO ---
def run_simulation(alpha, beta, phi):
    """
    Risolve il sistema e traccia il grafico al variare dei parametri interattivi.
    """
    # Condizioni iniziali del sistema [h0, s0]
    initial_state = [0.8, 0.1]

    # Discretizzazione temporale (da t=0 a t=100)
    t_span = (0, 100)
    t_eval = np.linspace(t_span[0], t_span[1], 1000)

    # Calcolo dell'integrale definito tramite Runge-Kutta
    solution = solve_ivp(
        creative_loop_system,
        t_span,
        initial_state,
        args=(alpha, beta, phi),
        t_eval=t_eval,
        method='RK45'
    )

    # Estrazione vettori risultati
    t = solution.t
    h_vals = solution.y[0]
    s_vals = solution.y[1]

    # Ricalcolo rigoroso della Soglia Critica di Iniezione (phi_crit)
    phi_crit = beta * h_tau * K_s - alpha * h_tau * (1 - h_tau / K)
    # Clamp visivo a zero se l'autorigenerazione è sufficiente senza iniezione esterna
    phi_crit_display = max(0, phi_crit)

    # Inizializzazione della figura
    plt.figure(figsize=(12, 6))

    # Tracciamento traiettorie di stato
    plt.plot(t, h_vals, label='Informazione Umana (h)', color='blue', linewidth=2.5)
    plt.plot(t, s_vals, label='Informazione Sintetica (s)', color='red', linewidth=2.5)

    # Tracciamento Soglia MAD
    plt.axhline(y=h_tau, color='black', linestyle='--', linewidth=1.5, label='Soglia MAD (h_tau = 0.1)')

    # Identificazione area di collasso autofago
    plt.fill_between(t, 0, h_tau, color='gray', alpha=0.2, label='Zona di Collasso (Autofagia)')

    # Analisi dello stato finale per l'allerta visiva
    if h_vals[-1] < h_tau:
        plt.title('STATO SISTEMA: COLLASSO AUTOFAGO IRREVERSIBILE', color='red', weight='bold', fontsize=14)
    else:
        plt.title('STATO SISTEMA: OMEOSTASI INFORMAZIONALE MANTENUTA', color='green', weight='bold', fontsize=14)

    # Formattazione assi e legenda
    plt.xlabel('Tempo (t)', fontsize=12)
    plt.ylabel('Volume Informazionale', fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.ylim(0, 1.4)
    plt.xlim(0, 100)

    # Box testuale con calcolo analitico in tempo reale
    info_text = (f"Parametri Attuali:\n"
                 f"α (Crescita) = {alpha:.2f}\n"
                 f"β (Attrito) = {beta:.2f}\n"
                 f"φ (Iniezione) = {phi:.3f}\n"
                 f"-----------------\n"
                 f"φ_crit (Soglia Teorica) = {phi_crit_display:.3f}")

    plt.text(102, 0.7, info_text, bbox=dict(facecolor='white', edgecolor='black', alpha=0.9), fontsize=11, family='monospace')

    # Gestione del layout spaziale per evitare sovrapposizioni
    plt.subplots_adjust(right=0.8)
    plt.legend(loc='upper right')
    plt.show()

# --- 4. ISTANZIAZIONE INTERFACCIA WIDGET (COLAB) ---
# Costruzione formale degli slider secondo i range prestabiliti
alpha_slider = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='α (Crescita Umana):', style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'))
beta_slider = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description='β (Interferenza IA):', style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'))
phi_slider = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='φ (Iniezione Esterna):', style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'))

# Binding della funzione di simulazione ai widget interattivi
interactive_plot = widgets.interactive(run_simulation, alpha=alpha_slider, beta=beta_slider, phi=phi_slider)
display(interactive_plot)